# 📘 MODULE 3 · Assignment M03.4: Descriptive Statistics (10 pts)
**Due in Module 3.** When you finish, upload your PDF to the Canvas assignment named **M03.4**.

**This week you:** pick 3 variables that relate to YOUR research question, summarize them, chart them, and interpret.

**Same 5 steps as every week:** 1 run setup → 2 fill in your project card → 3 load data → 4 do this week's TODO cells → 5 run the submission block and export the PDF.

**You're done when:** all self-checks are ✅, the stats and charts cells show output, and the 📤 submission block prints cleanly.

🧰 **First time here?** You will not write any Python. Press ▶ on a cell to run it, and type words between the quotation marks in 🟨 TODO cells. 🟥 Red error text can't hurt anything. 🟨 Yellow "NOT DONE YET" is coaching, not an error. `Runtime → Run all` fixes most weird problems. Stuck for 15 minutes? Screenshot it and message me.

**Before anything else:** `File → Save a copy in Drive`, then rename it `HIM4331_M03.4_LastnameFirstname`.

## ▶ Step 1: setup (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 1: tools and self-check helpers (just run it)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import datetime

pd.set_option("display.max_columns", 60)

_CHECKS = {}   # every self-check records its latest result here; the submission block reports them

def check(condition, ok_msg, fix_msg):
    """Friendly self-check: green check if done, yellow hint if not."""
    _CHECKS[fix_msg] = (bool(condition), ok_msg)
    if condition:
        print("✅ " + ok_msg)
        return True
    else:
        print("🟨 NOT DONE YET → " + fix_msg)
        return False

def banner(module, step, points):
    print("=" * 62)
    print(f"📤 MODULE {module} | {step} SUBMISSION BLOCK  ({points} pts)")
    print("=" * 62)
    print(f"Student:              {STUDENT_NAME}")
    print(f"Dataset:              {DATASET_CHOICE}")
    print(f"My research question: {MY_RESEARCH_QUESTION}")
    print(f"My stakeholder:       {MY_STAKEHOLDER}")
    print(f"Generated:            {datetime.datetime.now():%Y-%m-%d %H:%M}")
    print("-" * 62)
    _n = len(_CHECKS)
    _fail = [(fix, ok) for fix, (passed, ok) in _CHECKS.items() if not passed]
    if _n and not _fail:
        print(f"SELF-CHECK STATUS: ✅ ALL {_n} CHECKS PASSED")
    else:
        print(f"SELF-CHECK STATUS: 🟨 {len(_fail)} OF {_n} NOT DONE")
    for fix, (passed, ok) in _CHECKS.items():
        print(("   ✅ " + ok) if passed else ("   🟨 " + fix))
    print("-" * 62)

def next_week_card():
    """Prints the exact lines to paste into next week's project card."""
    g = globals()
    print("-" * 62)
    print("📋 YOUR PROJECT CARD FOR NEXT WEEK (copy these lines into Step 2 next time):")
    print(f'    STUDENT_NAME         = "{STUDENT_NAME}"')
    print(f'    DATASET_CHOICE       = "{DATASET_CHOICE}"')
    print(f'    MY_RESEARCH_QUESTION = "{MY_RESEARCH_QUESTION}"')
    print(f'    MY_STAKEHOLDER       = "{MY_STAKEHOLDER}"')
    if g.get("M04_COLUMN") and g.get("M04_FIX"):
        print(f'    M04_COLUMN = "{g["M04_COLUMN"]}"')
        print(f'    M04_FIX    = "{g["M04_FIX"]}"')
        if g.get("FLAG_FROM_COLUMN"):
            print(f'    FLAG_FROM_COLUMN = "{g["FLAG_FROM_COLUMN"]}"')
            print(f'    FLAG_ABOVE       = "{g.get("FLAG_ABOVE", "")}"')
            print(f'    FLAG_YES_LABEL   = "{g.get("FLAG_YES_LABEL", "")}"')
            print(f'    FLAG_NO_LABEL    = "{g.get("FLAG_NO_LABEL", "")}"')

# Where the four datasets live (your instructor hosts these; do not edit)
DATA_URLS = {
    "diabetes":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/diabetic_data.csv",
    "hospitals":    "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_hospitals.csv",
    "readmissions": "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_readmissions.csv",
    "breaches":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/hhs_breaches.csv",
}
_MISS_CODES = ["?", "Unknown", "UNKNOWN", "Not Available", "N/A", "NA", "None", ""]
print("✅ Step 1 done. Tools loaded. Go to Step 2.")

## 🟨 Step 2: your project card
Open last week's PDF, find the box titled 📋 YOUR PROJECT CARD FOR NEXT WEEK, and paste those lines over the blank ones below. Then press ▶. Your question is allowed to get sharper as you learn; if it changed, use the newer version.

In [ ]:
#@title 🟨 Step 2: fill in your project card
try:
    STUDENT_NAME         = ""   # e.g. "Jordan Rivera"
    DATASET_CHOICE       = ""   # one of: "diabetes", "hospitals", "readmissions", "breaches"
    MY_RESEARCH_QUESTION = ""   # e.g. "Do longer hospital stays predict 30-day readmission for diabetic patients?"
    MY_STAKEHOLDER       = ""   # who you're writing for, e.g. "Quality Director at a 300-bed hospital"

    # ---- self-checks (don't edit below) ----
    DATASET_CHOICE = DATASET_CHOICE.strip().lower()
    check(len(STUDENT_NAME.strip()) > 2, "Name entered.", "Type your name in STUDENT_NAME.")
    check(DATASET_CHOICE in DATA_URLS, f"Dataset '{DATASET_CHOICE}' is valid.",
          "DATASET_CHOICE must be exactly one of: diabetes, hospitals, readmissions, breaches")
    check(len(MY_RESEARCH_QUESTION.strip()) > 15 and MY_RESEARCH_QUESTION.strip().endswith("?"),
          "Research question entered.",
          "Write your research question as one full sentence ending in '?'")
    check(len(MY_STAKEHOLDER.strip()) > 5, "Stakeholder entered.",
          "Name a real decision-maker role, e.g. 'HIM privacy officer'.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## ▶ Step 3: load your data (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 3: load your dataset (just run it)
try:
    try:
        df = pd.read_csv(DATA_URLS[DATASET_CHOICE], low_memory=False)
        df_clean = df.copy()
        print(f"✅ Step 3 done. Loaded '{DATASET_CHOICE}': {df.shape[0]:,} rows × {df.shape[1]} columns")
    except KeyError:
        print("🟨 Fix DATASET_CHOICE in Step 2 first, then re-run this cell.")
    except Exception as e:
        print("🟨 Could not load the data. Check your internet connection and re-run.")
        print("   If it still fails, screenshot this and message your instructor:", repr(e))

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 🟨 Step 4: this week's work
💡 **Variable ideas for your dataset.** These are the exact column names in your file, copy them as-is:

* **diabetes** → numbers: `time_in_hospital`, `num_medications`, `num_lab_procedures`, `num_procedures`, `number_diagnoses` · categories: `age`, `race`, `gender`, `readmitted`, `insulin`, `change`, `diabetesMed`, `medical_specialty`
* **hospitals** → categories: `Hospital overall rating`, `Hospital Ownership`, `Hospital Type`, `Emergency Services`, `State` · numbers: `Count of Facility MORT Measures`, `Count of READM Measures Worse`
   * ⚠️ `Hospital overall rating` holds star ratings 1 to 5 but is stored as TEXT (some hospitals say `Not Available`). It will chart as a category here, which is fine for M03. M04 shows you how to turn it into a number.
* **readmissions** → numbers: `Excess Readmission Ratio`, `Number of Discharges`, `Predicted Readmission Rate`, `Expected Readmission Rate`, `Number of Readmissions` · categories: `Measure Name`, `State`
* **breaches** → numbers: `Individuals Affected` · categories: `Type of Breach`, `Covered Entity Type`, `Business Associate Present`, `Location of Breached Information`, `State`
   * ⚠️ `Individuals Affected` has one breach of 192.7 million people, so its histogram will look like a single bar on the far left. That is the outlier talking, not a broken chart. Mention it in your interpretation.

In [ ]:
#@title ▶ Remind me of my column names (just run it)
try:
    print("COLUMNS in your dataset (copy names exactly):")
    print(df.columns.tolist())

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title 🟨 TODO: pick your 3 variables (each in its own quotes, keep the commas)
try:
    MY_THREE_VARS = ["", "", ""]   # exact column names, each in its own quotes

    # ---- self-check ----
    MY_THREE_VARS = [v.strip() for v in MY_THREE_VARS]
    _missing = [v for v in MY_THREE_VARS if v not in df.columns]
    check(len(_missing) == 0 and len(set(MY_THREE_VARS)) == 3,
          "All 3 variables found in your dataset.",
          f"These aren't valid column names: {_missing or 'you have duplicates'}. "
          "Copy names EXACTLY (capitals, spaces, underscores) from the COLUMNS list above.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title ▶ Descriptive stats for your 3 variables (just run it)
try:
    for v in MY_THREE_VARS:
        print("─" * 50)
        print(f"VARIABLE: {v}")
        if pd.api.types.is_numeric_dtype(df[v]):
            print(df[v].describe().round(2).to_string())
        else:
            print(df[v].value_counts().head(12).to_string())

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title ▶ One chart per variable (just run it; charts appear in your PDF automatically)
try:
    for v in MY_THREE_VARS:
        plt.figure(figsize=(7, 3.5))
        if pd.api.types.is_numeric_dtype(df[v]):
            df[v].dropna().plot(kind="hist", bins=30, edgecolor="white")
            plt.ylabel("Count")
        else:
            df[v].value_counts().head(10).plot(kind="bar")
            plt.ylabel("Count")
            plt.xticks(rotation=30, ha="right")
        plt.title(f"Distribution of {v}")
        plt.xlabel(v)
        plt.tight_layout()
        plt.show()

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

### ✍️ What a good interpretation looks like
Flu-clinic example (different data, same shape of thinking):

> "The typical visit involved a patient aged 45 to 60, and 38% of patients never returned for a second dose. Second-dose drop-off was far more common among patients with no insurance listed, which surprised me because the vaccine is free. For a clinic operations manager, this suggests reminder outreach should focus on uninsured patients first."

Pattern: **what's typical → what stood out → what it means for my stakeholder.**

In [ ]:
#@title 🟨 TODO: your 3-sentence interpretation (finish the starters, or write your own)
MY_INTERPRETATION_M03 = """
The typical value/most common category I found was ...
One thing that stood out or surprised me was ...
For my stakeholder, this matters because ...
"""

check("I found was ..." not in MY_INTERPRETATION_M03
      and "surprised me was ..." not in MY_INTERPRETATION_M03
      and "matters because ..." not in MY_INTERPRETATION_M03,
      "Interpretation written.", "Finish all three sentence starters with your own words.")

## 📤 Step 5: submission block

In [ ]:
#@title 📤 M03.4 SUBMISSION BLOCK (Module 3): run, then File → Print → Save as PDF
try:
    banner(3, "M03.4 Descriptive Statistics", 10)
    print("1) MY 3 VARIABLES:", MY_THREE_VARS)
    print()
    print("2) DESCRIPTIVE STATS:")
    for v in MY_THREE_VARS:
        print("─" * 50)
        print(f"VARIABLE: {v}")
        if pd.api.types.is_numeric_dtype(df[v]):
            print(df[v].describe().round(2).to_string())
        else:
            print(df[v].value_counts().head(12).to_string())
    print()
    print("3) CHARTS: see the charts cell above (they print in the same PDF).")
    print()
    print("4) MY INTERPRETATION:")
    print(MY_INTERPRETATION_M03.strip())
    next_week_card()
    print("=" * 62)
    print("📎 Now: File → Print → Save as PDF → upload to the M03.4 Canvas assignment (Module 3).")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 📎 Step 5: export and submit
1. Make sure the submission block above printed with **no 🟨 warnings**.
2. `File → Print` (or Ctrl+P / Cmd+P) → choose **Save as PDF** → save it as `HIM4331_M03.4_LastnameFirstname.pdf`.
3. Upload that PDF to the Canvas assignment named **M03.4** (Module 3).
4. Keep this notebook. You will copy your project card from it next week.